# PyTorch Broadcasting — 45 Beginner-First Exercises

> **Working copy:** This file may contain learner answers and scratch work. Use the paired `_virgin.ipynb` notebook whenever you want a clean retry.

**Mastery target:** Repeated practice should make you able to read tensor axes, decide whether an elementwise operation is valid, predict its result shape, and prepare semantic axes deliberately for real PyTorch code.

## What you actually need to derive mentally

For everyday PyTorch, the essential skills are:

1. name what every axis means;
2. know each operand's actual shape;
3. decide whether the operation is compatible;
4. predict the result shape;
5. insert size-one axes when the current shape does not clearly express what each value belongs to.

An **aligned shape** is a temporary paper tool for difficult compatibility questions. It is not a hidden tensor PyTorch asks you to create. This notebook teaches alignment in one dedicated section and then stops requiring it everywhere.


## How to work

1. Run the setup cell after starting or restarting the kernel.
2. Work in order; concepts are introduced before they are tested.
3. Read the explicit fixture values and name the axes.
4. Predict only the fields requested by the current exercise.
5. Write the smallest direct PyTorch expression.
6. Run the private test and revisit the nearby explanation if it fails.


## Beginner glossary

You do not need prior PyTorch terminology. These words are used throughout the notebook:

- **Tensor:** a PyTorch container holding one number or a rectangular collection of numbers.
- **Shape:** a Python tuple giving the number of positions along each direction of a tensor. For example, `(2, 3)` means two rows and three columns.
- **Axis** (plural: **axes**): one direction of a tensor. Axes are numbered from left to right starting at `0`. In a matrix, axis `0` runs through rows and axis `1` runs through columns.
- **Operand:** an input used by an operator. In `a + b`, both `a` and `b` are operands.
- **Operator:** the action applied to operands, such as `+`, `-`, or `*`.
- **Elementwise operation:** an operation that combines corresponding positions. `a * b` is elementwise multiplication; `a @ b` is matrix multiplication and follows different rules.
- **Scalar:** a tensor containing one number and no axes. Its shape is `()`.
- **Vector:** a tensor with one axis. A three-value vector has shape `(3,)`.
- **Matrix:** a tensor with two axes, usually rows and columns.
- **Rank:** the number of axes. A scalar has rank zero, a vector rank one, and a matrix rank two.
- **Singleton axis:** an axis whose size is `1`. It contains one position along that direction, so broadcasting may reuse its value.
- **Broadcasting:** PyTorch's rule for making compatible operands behave as if they had the same shape before an elementwise operator is applied.
- **Compatible shapes:** shapes that PyTorch can broadcast together. When shapes are compared from the right, each size pair must be equal or one size must be `1`.
- **Aligned shape:** a paper-only comparison shape obtained by adding conceptual size-one positions on the left. It does not reshape the real tensor.
- **View:** a tensor with different shape or stride information that still refers to the same stored data.
- **Storage:** the memory containing a tensor's actual values.
- **Stride:** the step through storage used to move one position along an axis. A broadcasted view can use stride zero to reread one stored value.
- **Dtype:** the kind of value stored, such as floating-point numbers (`DTYPE`) or Boolean values (`torch.bool`).
- **Reduction:** an operation such as `sum` or `mean` that combines several values into fewer values.
- **Batch:** a group of examples processed together.
- **Feature:** one measured or learned value belonging to an example or token.
- **Token:** one position in a sequence.
- **Attention head:** one parallel attention calculation. **Query** means the position asking for information; **key** means a candidate position it may attend to.
- **Bias or offset:** a value added to another value. A feature bias supplies one added value per feature.
- **Mask:** values indicating which positions to keep, remove, allow, or forbid. Boolean masks use `True` and `False`.
- **Embedding:** a vector of feature values representing something such as a token or position.
- **Normalization:** recentering and rescaling values using statistics such as mean and variance.
- **Capstone/final practice:** a multi-step exercise combining several earlier ideas.

Every exercise also repeats the most important definitions and states what its axes mean.


## Broadcasting in plain language: make the operands operator-ready

Most exercises in this notebook practice the same five-step idea:

1. **Read each operand's actual shape.**
2. **Align the shape tuples from the right** so you can compare the same axis positions.
3. **Determine one common result shape.** At each aligned axis, sizes must be equal or one size must be `1`.
4. **Write the operator-ready tensors.** Each operand behaves as if its values were expanded to the common shape.
5. **Apply the operator element by element** to operands that now logically have the same shape.

The important distinction is:

- **Alignment** changes only how you write shapes for comparison.
- **Broadcasting/expansion** makes values logically behave at the common shape.

### Practical tip: expand from the last axis toward the first

When several axes must expand, do not try to imagine the complete tensor in one jump. Use this repeatable manual procedure:

1. Write the current aligned shape above the common target shape.
2. Start with the **last (rightmost) axis**.
3. If its current size is `1`, repeat each value across that axis until it reaches the target size.
4. Move one axis left. If that size is `1`, repeat the complete block you just built.
5. Skip axes whose sizes already match, and continue until you reach the first axis.

This order also matches the nesting of a tensor literal: expanding the last axis duplicates values inside the innermost lists; expanding the next axis duplicates those completed lists; expanding an outer axis duplicates larger completed blocks.

For example:

```text
current: (1, 3, 1, 1)
target:  (2, 3, 2, 2)

width:   (1, 3, 1, 1) → (1, 3, 1, 2)
height:  (1, 3, 1, 2) → (1, 3, 2, 2)
channel: already 3, so do not expand it
batch:   (1, 3, 2, 2) → (2, 3, 2, 2)
```

#### Value-level example

Suppose one prepared tensor contains one value for each of two rows. Its axes are `(batch, row, column)`, and its current shape is `(1, 2, 1)`:

```python
prepared = torch.tensor(
    [
        [
            [10.0],
            [20.0],
        ],
    ],
    dtype=DTYPE,
)
```

The target shape is `(2, 2, 3)`. Expand the last axis first, changing the single column into three columns:

```python
after_column_expansion = torch.tensor(
    [
        [
            [10.0, 10.0, 10.0],
            [20.0, 20.0, 20.0],
        ],
    ],
    dtype=DTYPE,
)
```

The row axis already has size `2`, so skip it. Then expand the first axis by repeating the completed two-row block for both batch items:

```python
operator_ready = torch.tensor(
    [
        [
            [10.0, 10.0, 10.0],
            [20.0, 20.0, 20.0],
        ],
        [
            [10.0, 10.0, 10.0],
            [20.0, 20.0, 20.0],
        ],
    ],
    dtype=DTYPE,
)
```

The visible path is therefore:

```text
(1, 2, 1) → (1, 2, 3) → (2, 2, 3)
```

The order does not change the final mathematical result, and PyTorch does not need to allocate each intermediate tensor. It is a reasoning and formatting technique that makes multi-axis expansion easier to verify.

### Example 1: scalar scale times a matrix

Suppose `scale` contains `2.0` and `matrix` has shape `(2, 3)`:

```text
scale actual shape:          ()
scale aligned shape:     (1, 1)
common result shape:     (2, 3)
scale operator-ready:    (2, 3)
```

The operator-ready values are:

```python
scale_before_op = torch.tensor(
    [[2.0, 2.0, 2.0],
     [2.0, 2.0, 2.0]],
    dtype=DTYPE,
)
```

The multiplication can now be understood position by position against the `(2, 3)` matrix.

### Example 2: one value per column

A vector with shape `(3,)` aligns with the final matrix axis:

```text
vector actual shape:       (3,)
vector aligned shape:    (1, 3)
common result shape:     (2, 3)
vector operator-ready:   (2, 3)
```

For values `[10, 20, 30]`, the operator-ready tensor is:

```python
vector_before_op = torch.tensor(
    [[10.0, 20.0, 30.0],
     [10.0, 20.0, 30.0]],
    dtype=DTYPE,
)
```

### Example 3: one value per row

A column tensor with shape `(2, 1)` already has row and column axes, but its size-one column axis must grow:

```text
column actual/aligned shape: (2, 1)
common result shape:         (2, 3)
column operator-ready:       (2, 3)
```

For row values `10` and `20`, it behaves as:

```python
column_before_op = torch.tensor(
    [[10.0, 10.0, 10.0],
     [20.0, 20.0, 20.0]],
    dtype=DTYPE,
)
```

This is the mental model used by the explicit `*_before_op` fields. PyTorch usually reuses stored values through shape and stride rules instead of allocating these full repeated tensors, but the operator behaves as if these operator-ready tensors existed.

Two exceptions are also important:

- If operands already have the same shape, they are already operator-ready and no expansion is needed.
- If aligned sizes conflict and neither size is `1`, no common operator-ready shape exists and the operation is invalid.


## Beginner checklist

Before an elementwise operation, ask:

1. What is each actual shape?
2. What does each axis mean?
3. Are the shapes already equal?
4. If not, is a scalar or size-one axis clearly being reused?
5. If intent is unclear, should I insert a singleton axis with `None` or `unsqueeze`?
6. Only in the alignment section: how do the shapes right-align?
7. Only in the expansion section: which aligned size-one axes are reused?


## Course map

1. Shapes before broadcasting — Exercises 001–008
2. Value reuse before formal rules — Exercises 009–016
3. Singleton axes express intent — Exercises 017–024
4. Alignment as a dedicated reasoning tool — Exercises 025–027
5. Shape preparation and multi-operand broadcasting — Exercises 028–029
6. Machine-learning axis patterns — Exercises 030–037
7. Explicit APIs, storage, and capstones — Exercises 038–045


In [1]:
# Supplied private-test infrastructure: run once after starting or restarting the kernel.
import importlib.util as _fixture_importlib
from pathlib import Path as _FixturePath

import torch

DTYPE = torch.float64
_REFS = {}
_MISSING = object()

_fixture_filename = "_torch_broadcasting_fixtures.py"
_fixture_relatives = (
    _FixturePath(_fixture_filename),
    _FixturePath("notebooks") / _fixture_filename,
)
_fixture_path = next(
    (
        base / relative
        for base in (_FixturePath.cwd(), *_FixturePath.cwd().parents)
        for relative in _fixture_relatives
        if (base / relative).is_file()
    ),
    None,
)
if _fixture_path is None:
    raise FileNotFoundError(f"Could not find {_fixture_filename}. Keep it beside the notebook.")
_fixture_spec = _fixture_importlib.spec_from_file_location("_torch_broadcasting_fixtures", _fixture_path)
assert _fixture_spec is not None and _fixture_spec.loader is not None
_fixture_module = _fixture_importlib.module_from_spec(_fixture_spec)
_fixture_spec.loader.exec_module(_fixture_module)


def _register_case(key, tensors):
    _REFS[key] = _fixture_module.build_reference(key, tensors)


def _check_private_value(variable_name, key, field):
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    expected = _REFS[key][field]
    assert type(actual) is type(expected), f"`{variable_name}` must have type {type(expected).__name__}."
    assert actual == expected, f"`{variable_name}` is not correct; revisit the nearby reasoning steps."
    print(f"PASS: {variable_name}")


def _check_private_tensor(variable_name, key, field):
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, torch.Tensor), f"`{variable_name}` must be a torch.Tensor."
    expected = _REFS[key][field]
    assert actual.shape == expected.shape, f"`{variable_name}` has the wrong shape."
    assert actual.dtype == expected.dtype, f"`{variable_name}` has the wrong dtype."
    try:
        torch.testing.assert_close(actual, expected, rtol=1e-7, atol=1e-9)
    except AssertionError:
        raise AssertionError(f"`{variable_name}` has the right shape but incorrect values.") from None
    print(f"PASS: {variable_name}")


print("Broadcasting helpers ready. Start at Exercise 001.")


C:\Users\giloz\dev\pytorch_master_through_exercises\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Broadcasting helpers ready. Start at Exercise 001.


## 1. Shapes before broadcasting

Learn scalar, vector, matrix, rank, axes, and equal-shape elementwise operations. No alignment or expansion vocabulary appears here.

This section covers Exercises 001–008.


### Exercise 001 — Scalar plus scalar

**Purpose:** Recognize a scalar tensor: one number with no axes and shape `()`.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict both actual shapes and the result shape, then add the scalars.

**Ingredients:** A scalar tensor has no axes, rank zero, and shape `()`.

**Axis meaning in this exercise:** Both operands are scalars, so they have no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Equal vectors add elementwise


In [2]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(2.0, dtype=DTYPE)
b = torch.tensor(-0.5, dtype=DTYPE)
_register_case("ex001", {'a': a, 'b': b})


In [3]:
# Exercise 001: complete only the fields requested above; do not use autograd.
# Define `ex001_a_shape`.
# Define `ex001_b_shape`.
# Define `ex001_out_shape`.
# Define `ex001_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex001_a_shape = ()
ex001_b_shape = ()
ex001_out_shape = ()
ex001_out = torch.tensor(1.5, dtype=DTYPE)


In [4]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex001_a_shape", "ex001", "a_shape")
_check_private_value("ex001_b_shape", "ex001", "b_shape")
_check_private_value("ex001_out_shape", "ex001", "out_shape")
_check_private_tensor("ex001_out", "ex001", "out")


PASS: ex001_a_shape
PASS: ex001_b_shape
PASS: ex001_out_shape
PASS: ex001_out


### Exercise 002 — Equal vectors add elementwise

**Purpose:** Read a vector shape and add values at matching positions.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict shapes, then add matching vector entries.

**Ingredients:** A length-three vector has shape `(3,)`; the comma makes it a Python tuple.

**Axis meaning in this exercise:** For both vectors, axis `0` lists the three vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Equal vectors multiply elementwise


In [5]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
b = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex002", {'a': a, 'b': b})


In [6]:
# Exercise 002: complete only the fields requested above; do not use autograd.
# Define `ex002_a_shape`.
# Define `ex002_b_shape`.
# Define `ex002_out_shape`.
# Define `ex002_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex002_a_shape = (3,)
ex002_b_shape = (3,)
ex002_out_shape = (3,)
ex002_out = torch.tensor([11.0, 22.0, 33.0], dtype=DTYPE)


In [7]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex002_a_shape", "ex002", "a_shape")
_check_private_value("ex002_b_shape", "ex002", "b_shape")
_check_private_value("ex002_out_shape", "ex002", "out_shape")
_check_private_tensor("ex002_out", "ex002", "out")


PASS: ex002_a_shape
PASS: ex002_b_shape
PASS: ex002_out_shape
PASS: ex002_out


### Exercise 003 — Equal vectors multiply elementwise

**Purpose:** Learn that `*` multiplies matching positions; it is different from matrix multiplication with `@`.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Predict shapes, then multiply matching vector entries.

**Ingredients:** The `*` operator multiplies corresponding positions.

**Axis meaning in this exercise:** For both vectors, axis `0` lists the three vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Equal matrices add elementwise


In [8]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([2.0, -1.0, 4.0], dtype=DTYPE)
b = torch.tensor([0.5, 3.0, -2.0], dtype=DTYPE)
_register_case("ex003", {'a': a, 'b': b})


In [9]:
# Exercise 003: complete only the fields requested above; do not use autograd.
# Define `ex003_a_shape`.
# Define `ex003_b_shape`.
# Define `ex003_out_shape`.
# Define `ex003_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex003_a_shape = (3,)
ex003_b_shape = (3,)
ex003_out_shape = (3,)
ex003_out = torch.tensor([1.0, -3.0, -8.0], dtype=DTYPE)


In [10]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex003_a_shape", "ex003", "a_shape")
_check_private_value("ex003_b_shape", "ex003", "b_shape")
_check_private_value("ex003_out_shape", "ex003", "out_shape")
_check_private_tensor("ex003_out", "ex003", "out")


PASS: ex003_a_shape
PASS: ex003_b_shape
PASS: ex003_out_shape
PASS: ex003_out


### Exercise 004 — Equal matrices add elementwise

**Purpose:** Read matrix rows and columns directly from an explicit tensor.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict both matrix shapes and add matching entries.

**Ingredients:** For shape `(2, 3)`, axis `0` has two rows and axis `1` has three columns.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Equal matrices multiply elementwise


In [11]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
b = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]], dtype=DTYPE)
_register_case("ex004", {'a': a, 'b': b})


In [12]:
# Exercise 004: complete only the fields requested above; do not use autograd.
# Define `ex004_a_shape`.
# Define `ex004_b_shape`.
# Define `ex004_out_shape`.
# Define `ex004_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex004_a_shape = (2, 3)
ex004_b_shape = (2, 3)
ex004_out_shape = (2, 3)
ex004_out = torch.tensor(
    [
        [11.0, 22.0, 33.0],
        [44.0, 55.0, 66.0],
    ],
    dtype=DTYPE,
)


In [13]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex004_a_shape", "ex004", "a_shape")
_check_private_value("ex004_b_shape", "ex004", "b_shape")
_check_private_value("ex004_out_shape", "ex004", "out_shape")
_check_private_tensor("ex004_out", "ex004", "out")


PASS: ex004_a_shape
PASS: ex004_b_shape
PASS: ex004_out_shape
PASS: ex004_out


### Exercise 005 — Equal matrices multiply elementwise

**Purpose:** Confirm that equal matrix shapes need no value reuse.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Predict shapes, then multiply matching matrix entries.

**Ingredients:** This is elementwise multiplication, not `a @ b`.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** One-row matrix


In [14]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=DTYPE)
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]], dtype=DTYPE)
_register_case("ex005", {'a': a, 'b': b})


In [15]:
# Exercise 005: complete only the fields requested above; do not use autograd.
# Define `ex005_a_shape`.
# Define `ex005_b_shape`.
# Define `ex005_out_shape`.
# Define `ex005_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex005_a_shape = (2, 2)
ex005_b_shape = (2, 2)
ex005_out_shape = (2, 2)
ex005_out = torch.tensor(
    [
        [5.0, 12.0],
        [21.0, 32.0],
    ],
    dtype=DTYPE,
)


In [16]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex005_a_shape", "ex005", "a_shape")
_check_private_value("ex005_b_shape", "ex005", "b_shape")
_check_private_value("ex005_out_shape", "ex005", "out_shape")
_check_private_tensor("ex005_out", "ex005", "out")


PASS: ex005_a_shape
PASS: ex005_b_shape
PASS: ex005_out_shape
PASS: ex005_out


### Exercise 006 — One-row matrix

**Purpose:** See why a one-row matrix has two axes while a vector has one axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a - b`

**Task:** Predict the two-axis shapes, then subtract.

**Ingredients:** Double brackets create a matrix with one row; its shape has two entries.

**Axis meaning in this exercise:** For both one-row matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** One-element vector


In [17]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
b = torch.tensor([[4.0, 5.0, 6.0]], dtype=DTYPE)
_register_case("ex006", {'a': a, 'b': b})


In [18]:
# Exercise 006: complete only the fields requested above; do not use autograd.
# Define `ex006_a_shape`.
# Define `ex006_b_shape`.
# Define `ex006_out_shape`.
# Define `ex006_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex006_a_shape = (1, 3)
ex006_b_shape = (1, 3)
ex006_out_shape = (1, 3)
ex006_out = torch.tensor(
    [
        [-3.0, -3.0, -3.0],
    ],
    dtype=DTYPE,
)


In [19]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex006_a_shape", "ex006", "a_shape")
_check_private_value("ex006_b_shape", "ex006", "b_shape")
_check_private_value("ex006_out_shape", "ex006", "out_shape")
_check_private_tensor("ex006_out", "ex006", "out")


PASS: ex006_a_shape
PASS: ex006_b_shape
PASS: ex006_out_shape
PASS: ex006_out


### Exercise 007 — One-element vector

**Purpose:** Distinguish shape `(1,)` from scalar shape `()`.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict the vector shapes and result.

**Ingredients:** Each tensor has one axis containing one element; neither tensor is a scalar.

**Axis meaning in this exercise:** For both vectors, axis `0` lists vector positions; each vector happens to contain one position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Equal three-axis tensors


In [20]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([3.0], dtype=DTYPE)
b = torch.tensor([2.0], dtype=DTYPE)
_register_case("ex007", {'a': a, 'b': b})


In [21]:
# Exercise 007: complete only the fields requested above; do not use autograd.
# Define `ex007_a_shape`.
# Define `ex007_b_shape`.
# Define `ex007_out_shape`.
# Define `ex007_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex007_a_shape = (1,)
ex007_b_shape = (1,)
ex007_out_shape = (1,)
ex007_out = torch.tensor([5.0], dtype=DTYPE)

In [22]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex007_a_shape", "ex007", "a_shape")
_check_private_value("ex007_b_shape", "ex007", "b_shape")
_check_private_value("ex007_out_shape", "ex007", "out_shape")
_check_private_tensor("ex007_out", "ex007", "out")


PASS: ex007_a_shape
PASS: ex007_b_shape
PASS: ex007_out_shape
PASS: ex007_out


### Exercise 008 — Equal three-axis tensors

**Purpose:** Apply the same position-by-position addition to tensors with three axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict all three axes, then add matching entries.

**Ingredients:** Name the axes `(batch, rows, columns)` for this exercise. Equal shapes still pair position by position.

**Axis meaning in this exercise:** For both tensors, axes are `(batch, rows, columns)` in that order.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Scalar reused across a vector


In [23]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(
    [
        [
            [1.0, 2.0],
            [3.0, 4.0]
        ],
    ], dtype=DTYPE
)
b = torch.tensor(
    [
        [
            [10.0, 20.0],
            [30.0, 40.0]
        ]
    ], dtype=DTYPE
)
_register_case("ex008", {'a': a, 'b': b})


In [24]:
# Exercise 008: complete only the fields requested above; do not use autograd.
# Define `ex008_a_shape`.
# Define `ex008_b_shape`.
# Define `ex008_out_shape`.
# Define `ex008_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex008_a_shape = (1, 2, 2)
ex008_b_shape = (1, 2, 2)
ex008_out_shape = (1, 2, 2)
ex008_out = torch.tensor([[[11.0, 22.0], [33.0, 44.0]]], dtype=DTYPE)

In [25]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex008_a_shape", "ex008", "a_shape")
_check_private_value("ex008_b_shape", "ex008", "b_shape")
_check_private_value("ex008_out_shape", "ex008", "out_shape")
_check_private_tensor("ex008_out", "ex008", "out")


PASS: ex008_a_shape
PASS: ex008_b_shape
PASS: ex008_out_shape
PASS: ex008_out


## 2. Value reuse before formal rules

See scalar and singleton values reused through visible tensors before learning PyTorch's mechanical alignment algorithm.

This section covers Exercises 009–016.


### Exercise 009 — Scalar reused across a vector

**Purpose:** See broadcasting as PyTorch reusing a smaller tensor's values before learning the formal comparison steps.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict actual and result shapes, then add the scalar to every vector entry.

**Ingredients:** Think of the scalar as available at each vector position. Do not write aligned shapes yet.

**Axis meaning in this exercise:** `a` is a scalar with no axes; axis `0` of `b` lists vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Vector times scalar


In [26]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(10.0, dtype=DTYPE)
b = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
_register_case("ex009", {'a': a, 'b': b})


In [27]:
# Exercise 009: complete only the fields requested above; do not use autograd.
# Define `ex009_a_shape`.
# Define `ex009_b_shape`.
# Define `ex009_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex009_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex009_out_shape`.
# Define `ex009_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex009_a_shape = ()
ex009_b_shape = (3,)
ex009_a_before_op = torch.tensor([10.0, 10.0, 10.0], dtype=DTYPE)
ex009_b_before_op = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)  # doesn't need to change
ex009_out_shape = (3,)
ex009_out = torch.tensor([11.0, 12.0, 13.0], dtype=DTYPE)

In [28]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex009_a_shape", "ex009", "a_shape")
_check_private_value("ex009_b_shape", "ex009", "b_shape")
_check_private_tensor("ex009_a_before_op", "ex009", "a_before_op")
_check_private_tensor("ex009_b_before_op", "ex009", "b_before_op")
_check_private_value("ex009_out_shape", "ex009", "out_shape")
_check_private_tensor("ex009_out", "ex009", "out")


PASS: ex009_a_shape
PASS: ex009_b_shape
PASS: ex009_a_before_op
PASS: ex009_b_before_op
PASS: ex009_out_shape
PASS: ex009_out


### Exercise 010 — Vector times scalar

**Purpose:** See that a scalar is reused even when the vector is written before it.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Multiply each vector entry by the scalar.

**Ingredients:** The vector is before `*` and the scalar is after it; vector values are not reversed.

**Axis meaning in this exercise:** Axis `0` of `a` lists vector positions; `b` is a scalar with no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Scalar reused across a matrix


In [29]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([1.0, -2.0, 3.0], dtype=DTYPE)
b = torch.tensor(4.0, dtype=DTYPE)
_register_case("ex010", {'a': a, 'b': b})


In [30]:
# Exercise 010: complete only the fields requested above; do not use autograd.
# Define `ex010_a_shape`.
# Define `ex010_b_shape`.
# Define `ex010_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex010_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex010_out_shape`.
# Define `ex010_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex010_a_shape = (3,)
ex010_b_shape = ()
ex010_a_before_op = torch.tensor([1.0, -2.0, 3.0], dtype=DTYPE)
ex010_b_before_op = torch.tensor([4.0, 4.0, 4.0], dtype=DTYPE)
ex010_out_shape = (3,)
ex010_out = torch.tensor([4.0, -8.0, 12.0], dtype=DTYPE)

In [31]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex010_a_shape", "ex010", "a_shape")
_check_private_value("ex010_b_shape", "ex010", "b_shape")
_check_private_tensor("ex010_a_before_op", "ex010", "a_before_op")
_check_private_tensor("ex010_b_before_op", "ex010", "b_before_op")
_check_private_value("ex010_out_shape", "ex010", "out_shape")
_check_private_tensor("ex010_out", "ex010", "out")


PASS: ex010_a_shape
PASS: ex010_b_shape
PASS: ex010_a_before_op
PASS: ex010_b_before_op
PASS: ex010_out_shape
PASS: ex010_out


### Exercise 011 — Scalar reused across a matrix

**Purpose:** Extend scalar reuse across rows and columns.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add the scalar to every matrix position.

**Ingredients:** The matrix values are explicit so you can focus on reuse rather than decoding a constructor.

**Axis meaning in this exercise:** For `a`, axis `0` means rows and axis `1` means columns; scalar `b` has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Matrix minus scalar


In [32]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
b = torch.tensor(0.5, dtype=DTYPE)
_register_case("ex011", {'a': a, 'b': b})


In [33]:
# Exercise 011: complete only the fields requested above; do not use autograd.
# Define `ex011_a_shape`.
# Define `ex011_b_shape`.
# Define `ex011_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex011_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex011_out_shape`.
# Define `ex011_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex011_a_shape = (2, 3)
ex011_b_shape = ()
ex011_a_before_op = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
ex011_b_before_op = torch.tensor([[0.5, 0.5, 0.5], [0.5, 0.5, 0.5]], dtype=DTYPE)
ex011_out_shape = (2, 3)
ex011_out = torch.tensor([[0.5, 1.5, 2.5], [3.5, 4.5, 5.5]], dtype=DTYPE)

In [34]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex011_a_shape", "ex011", "a_shape")
_check_private_value("ex011_b_shape", "ex011", "b_shape")
_check_private_tensor("ex011_a_before_op", "ex011", "a_before_op")
_check_private_tensor("ex011_b_before_op", "ex011", "b_before_op")
_check_private_value("ex011_out_shape", "ex011", "out_shape")
_check_private_tensor("ex011_out", "ex011", "out")


PASS: ex011_a_shape
PASS: ex011_b_shape
PASS: ex011_a_before_op
PASS: ex011_b_before_op
PASS: ex011_out_shape
PASS: ex011_out


### Exercise 012 — Matrix minus scalar

**Purpose:** Reverse the placement of matrix and scalar while preserving the same result shape.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a - b`

**Task:** Subtract the scalar from every matrix entry.

**Ingredients:** Subtraction values depend on operand order, but shape compatibility does not.

**Axis meaning in this exercise:** For `a`, axis `0` means rows and axis `1` means columns; scalar `b` has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** One stored value reused across a vector


In [35]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[2.0, 4.0], [6.0, 8.0]], dtype=DTYPE)
b = torch.tensor(1.0, dtype=DTYPE)
_register_case("ex012", {'a': a, 'b': b})


In [36]:
# Exercise 012: complete only the fields requested above; do not use autograd.
# Define `ex012_a_shape`.
# Define `ex012_b_shape`.
# Define `ex012_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex012_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex012_out_shape`.
# Define `ex012_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex012_a_shape = (2, 2)
ex012_b_shape = ()
ex012_a_before_op = torch.tensor([[2.0, 4.0], [6.0, 8.0]], dtype=DTYPE)
ex012_b_before_op = torch.tensor([[1.0, 1.0], [1.0, 1.0]], dtype=DTYPE)
ex012_out_shape = (2, 2)
ex012_out = torch.tensor([[1.0, 3.0], [5.0, 7.0]], dtype=DTYPE)

In [37]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex012_a_shape", "ex012", "a_shape")
_check_private_value("ex012_b_shape", "ex012", "b_shape")
_check_private_tensor("ex012_a_before_op", "ex012", "a_before_op")
_check_private_tensor("ex012_b_before_op", "ex012", "b_before_op")
_check_private_value("ex012_out_shape", "ex012", "out_shape")
_check_private_tensor("ex012_out", "ex012", "out")


PASS: ex012_a_shape
PASS: ex012_b_shape
PASS: ex012_a_before_op
PASS: ex012_b_before_op
PASS: ex012_out_shape
PASS: ex012_out


### Exercise 013 — One stored value reused across a vector

**Purpose:** See how a one-element vector supplies its value at every position of a longer vector.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add the one-element vector throughout the longer vector.

**Ingredients:** The compact operand has an actual axis of size `1`; this differs from scalar shape `()` even though both can be reused.

**Axis meaning in this exercise:** Axis `0` of each operand lists vector positions; `a` has only one such position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** One stored value reused across a matrix


In [38]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([2.0], dtype=DTYPE)
b = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex013", {'a': a, 'b': b})


In [39]:
# Exercise 013: complete only the fields requested above; do not use autograd.
# Define `ex013_a_shape`.
# Define `ex013_b_shape`.
# Define `ex013_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex013_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex013_out_shape`.
# Define `ex013_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex013_a_shape = (1,)
ex013_b_shape = (3,)
ex013_a_before_op = torch.tensor([2.0, 2.0, 2.0], dtype=DTYPE)
ex013_b_before_op = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
ex013_out_shape = (3,)
ex013_out = torch.tensor([12.0, 22.0, 32.0], dtype=DTYPE)

In [40]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex013_a_shape", "ex013", "a_shape")
_check_private_value("ex013_b_shape", "ex013", "b_shape")
_check_private_tensor("ex013_a_before_op", "ex013", "a_before_op")
_check_private_tensor("ex013_b_before_op", "ex013", "b_before_op")
_check_private_value("ex013_out_shape", "ex013", "out_shape")
_check_private_tensor("ex013_out", "ex013", "out")


PASS: ex013_a_shape
PASS: ex013_b_shape
PASS: ex013_a_before_op
PASS: ex013_b_before_op
PASS: ex013_out_shape
PASS: ex013_out


### Exercise 014 — One stored value reused across a matrix

**Purpose:** Use an explicit `(1, 1)` tensor throughout a matrix.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Multiply every matrix entry by the one stored value.

**Ingredients:** The compact tensor already has two axes, each of size `1`.

**Axis meaning in this exercise:** Both operands use axis `0` for rows and axis `1` for columns; `a` has size `1` on both axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Scalar reused across three axes


In [41]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[2.0]], dtype=DTYPE)
b = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
_register_case("ex014", {'a': a, 'b': b})


In [42]:
# Exercise 014: complete only the fields requested above; do not use autograd.
# Define `ex014_a_shape`.
# Define `ex014_b_shape`.
# Define `ex014_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex014_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex014_out_shape`.
# Define `ex014_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex014_a_shape = (1, 1)
ex014_b_shape = (2, 3)
ex014_a_before_op = torch.tensor([[2.0, 2.0, 2.0], [2.0, 2.0, 2.0]], dtype=DTYPE)
ex014_b_before_op = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
ex014_out_shape = (2, 3)
ex014_out = torch.tensor([[2.0, 4.0, 6.0], [8.0, 10.0, 12.0]], dtype=DTYPE)

In [43]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex014_a_shape", "ex014", "a_shape")
_check_private_value("ex014_b_shape", "ex014", "b_shape")
_check_private_tensor("ex014_a_before_op", "ex014", "a_before_op")
_check_private_tensor("ex014_b_before_op", "ex014", "b_before_op")
_check_private_value("ex014_out_shape", "ex014", "out_shape")
_check_private_tensor("ex014_out", "ex014", "out")


PASS: ex014_a_shape
PASS: ex014_b_shape
PASS: ex014_a_before_op
PASS: ex014_b_before_op
PASS: ex014_out_shape
PASS: ex014_out


### Exercise 015 — Scalar reused across three axes

**Purpose:** See that a scalar can be added to a tensor with any number of axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add the scalar throughout the rank-three tensor.

**Ingredients:** Axes are `(batch, rows, columns)`. The scalar contributes no semantic axis.

**Axis meaning in this exercise:** Axes of `a` are `(batch, rows, columns)`; scalar `b` has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Equal matrices require no reuse


In [44]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[[1.0, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]], dtype=DTYPE)
b = torch.tensor(0.25, dtype=DTYPE)
_register_case("ex015", {'a': a, 'b': b})


In [45]:
# Exercise 015: complete only the fields requested above; do not use autograd.
# Define `ex015_a_shape`.
# Define `ex015_b_shape`.
# Define `ex015_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex015_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex015_out_shape`.
# Define `ex015_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex015_a_shape = (2, 2, 2)
ex015_b_shape = ()
ex015_a_before_op = torch.tensor([[[1.0, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]], dtype=DTYPE)
ex015_b_before_op = torch.tensor([[[0.25, 0.25], [0.25, 0.25]], [[0.25, 0.25], [0.25, 0.25]]], dtype=DTYPE)
ex015_out_shape = (2, 2, 2)
ex015_out = torch.tensor([[[1.25, 2.25], [3.25, 4.25]], [[5.25, 6.25], [7.25, 8.25]]], dtype=DTYPE)

In [46]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex015_a_shape", "ex015", "a_shape")
_check_private_value("ex015_b_shape", "ex015", "b_shape")
_check_private_tensor("ex015_a_before_op", "ex015", "a_before_op")
_check_private_tensor("ex015_b_before_op", "ex015", "b_before_op")
_check_private_value("ex015_out_shape", "ex015", "out_shape")
_check_private_tensor("ex015_out", "ex015", "out")


PASS: ex015_a_shape
PASS: ex015_b_shape
PASS: ex015_a_before_op
PASS: ex015_b_before_op
PASS: ex015_out_shape
PASS: ex015_out


### Exercise 016 — Equal matrices require no reuse

**Purpose:** Compare broadcasting with the simpler case where both tensors already have the same shape.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add equal matrices and state the result shape.

**Ingredients:** Every position already has one matching value; no operand needs reuse.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Explicit row tensor


In [47]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=DTYPE)
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]], dtype=DTYPE)
_register_case("ex016", {'a': a, 'b': b})


In [48]:
# Exercise 016: complete only the fields requested above; do not use autograd.
# Define `ex016_a_shape`.
# Define `ex016_b_shape`.
# Define `ex016_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex016_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex016_out_shape`.
# Define `ex016_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex016_a_shape = (2, 2)
ex016_b_shape = (2, 2)
ex016_a_before_op = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=DTYPE)
ex016_b_before_op = torch.tensor([[10.0, 20.0], [30.0, 40.0]], dtype=DTYPE)
ex016_out_shape = (2, 2)
ex016_out = torch.tensor([[11.0, 22.0], [33.0, 44.0]], dtype=DTYPE)

In [49]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex016_a_shape", "ex016", "a_shape")
_check_private_value("ex016_b_shape", "ex016", "b_shape")
_check_private_tensor("ex016_a_before_op", "ex016", "a_before_op")
_check_private_tensor("ex016_b_before_op", "ex016", "b_before_op")
_check_private_value("ex016_out_shape", "ex016", "out_shape")
_check_private_tensor("ex016_out", "ex016", "out")


PASS: ex016_a_shape
PASS: ex016_b_shape
PASS: ex016_a_before_op
PASS: ex016_b_before_op
PASS: ex016_out_shape
PASS: ex016_out


## 3. Singleton axes express intent

Use row-shaped, column-shaped, and deliberately prepared tensors so axis meaning is visible in code.

This section covers Exercises 017–024.


### Exercise 017 — Explicit row tensor

**Purpose:** Attach three values to matrix columns with shape `(1, 3)`.

**Visible inputs:** `matrix`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + row`

**Task:** Add one value per column across both rows.

**Ingredients:** The row tensor's singleton axis `0` permits reuse down matrix rows.

**Axis meaning in this exercise:** For `matrix` and `row`, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Explicit column tensor


In [50]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex017", {'matrix': matrix, 'row': row})


In [51]:
# Exercise 017: complete only the fields requested above; do not use autograd.
# Define `ex017_matrix_shape`.
# Define `ex017_row_shape`.
# Define `ex017_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex017_row_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex017_out_shape`.
# Define `ex017_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex017_matrix_shape = (2, 3)
ex017_row_shape = (1, 3)
ex017_matrix_before_op = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
ex017_row_before_op = torch.tensor([[10.0, 20.0, 30.0], [10.0, 20.0, 30.0]], dtype=DTYPE)
ex017_out_shape = (2, 3)
ex017_out = torch.tensor([[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]], dtype=DTYPE)

In [52]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex017_matrix_shape", "ex017", "matrix_shape")
_check_private_value("ex017_row_shape", "ex017", "row_shape")
_check_private_tensor("ex017_matrix_before_op", "ex017", "matrix_before_op")
_check_private_tensor("ex017_row_before_op", "ex017", "row_before_op")
_check_private_value("ex017_out_shape", "ex017", "out_shape")
_check_private_tensor("ex017_out", "ex017", "out")


PASS: ex017_matrix_shape
PASS: ex017_row_shape
PASS: ex017_matrix_before_op
PASS: ex017_row_before_op
PASS: ex017_out_shape
PASS: ex017_out


### Exercise 018 — Explicit column tensor

**Purpose:** Attach two values to matrix rows with shape `(2, 1)`.

**Visible inputs:** `matrix`, `column`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + column`

**Task:** Add one value across every column of each row.

**Ingredients:** The column tensor's singleton axis `1` permits reuse across matrix columns.

**Axis meaning in this exercise:** For `matrix` and `column`, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Every row-column sum


In [53]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
column = torch.tensor([[10.0], [20.0]], dtype=DTYPE)
_register_case("ex018", {'matrix': matrix, 'column': column})


In [54]:
# Exercise 018: complete only the fields requested above; do not use autograd.
# Define `ex018_matrix_shape`.
# Define `ex018_column_shape`.
# Define `ex018_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex018_column_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex018_out_shape`.
# Define `ex018_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex018_matrix_shape = (2, 3)
ex018_column_shape = (2, 1)
ex018_matrix_before_op = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
ex018_column_before_op = torch.tensor([[10.0, 10.0, 10.0], [20.0, 20.0, 20.0]], dtype=DTYPE)
ex018_out_shape = (2, 3)
ex018_out = torch.tensor([[11.0, 12.0, 13.0], [24.0, 25.0, 26.0]], dtype=DTYPE)

In [55]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex018_matrix_shape", "ex018", "matrix_shape")
_check_private_value("ex018_column_shape", "ex018", "column_shape")
_check_private_tensor("ex018_matrix_before_op", "ex018", "matrix_before_op")
_check_private_tensor("ex018_column_before_op", "ex018", "column_before_op")
_check_private_value("ex018_out_shape", "ex018", "out_shape")
_check_private_tensor("ex018_out", "ex018", "out")


PASS: ex018_matrix_shape
PASS: ex018_column_shape
PASS: ex018_matrix_before_op
PASS: ex018_column_before_op
PASS: ex018_out_shape
PASS: ex018_out


### Exercise 019 — Every row-column sum

**Purpose:** Build a two-row, three-column result by combining a column tensor with a row tensor.

**Visible inputs:** `column`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `column + row`

**Task:** Create every column-value and row-value sum.

**Ingredients:** The column reuses values across columns; the row reuses values across rows.

**Axis meaning in this exercise:** For both tensors, axis `0` represents rows and axis `1` represents columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Prepare row-owned values


In [56]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
column = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex019", {'column': column, 'row': row})


In [57]:
# Exercise 019: complete only the fields requested above; do not use autograd.
# Define `ex019_column_shape`.
# Define `ex019_row_shape`.
# Define `ex019_column_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex019_row_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex019_out_shape`.
# Define `ex019_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex019_column_shape = (2, 1)
ex019_row_shape = (1, 3)
ex019_column_before_op = torch.tensor([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0]], dtype=DTYPE)
ex019_row_before_op = torch.tensor([[10.0, 20.0, 30.0], [10.0, 20.0, 30.0]], dtype=DTYPE)
ex019_out_shape = (2, 3)
ex019_out = torch.tensor([[11.0, 21.0, 31.0], [12.0, 22.0, 32.0]], dtype=DTYPE)

In [58]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex019_column_shape", "ex019", "column_shape")
_check_private_value("ex019_row_shape", "ex019", "row_shape")
_check_private_tensor("ex019_column_before_op", "ex019", "column_before_op")
_check_private_tensor("ex019_row_before_op", "ex019", "row_before_op")
_check_private_value("ex019_out_shape", "ex019", "out_shape")
_check_private_tensor("ex019_out", "ex019", "out")


PASS: ex019_column_shape
PASS: ex019_row_shape
PASS: ex019_column_before_op
PASS: ex019_row_before_op
PASS: ex019_out_shape
PASS: ex019_out


### Exercise 020 — Prepare row-owned values

**Purpose:** Add a size-one axis at the end so the vector clearly means one value per row.

**Visible inputs:** `rows`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `rows_col * grid`

**Task:** Create `rows_col` with `rows[:, None]`, then multiply it across columns.

**Ingredients:** Indexing with `None` inserts a real size-one axis; it does not copy values.

**Axis meaning in this exercise:** Axis `0` of `rows` means matrix rows. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `:` means keep every value on the existing axis. `None` means insert a new axis of size `1`. Therefore `rows[:, None]` keeps all row values and adds one column position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Prepare column-owned values


In [59]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
rows = torch.tensor([2.0, -1.0], dtype=DTYPE)
grid = torch.tensor([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], dtype=DTYPE)
_register_case("ex020", {'rows': rows, 'grid': grid})


In [60]:
# Exercise 020: complete only the fields requested above; do not use autograd.
# Define `ex020_rows_shape`.
# Define `ex020_grid_shape`.
# Define `ex020_rows_col_shape`.
# Define `ex020_rows_col` with PyTorch tensor operations.
# Define `ex020_rows_col_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex020_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex020_out_shape`.
# Define `ex020_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex020_rows_shape = (2,)
ex020_grid_shape = (2, 3)
ex020_rows_col_shape = (2, 1)
ex020_rows_col = rows[:, None]
ex020_rows_col_before_op = torch.tensor([[2.0, 2.0, 2.0], [-1.0, -1.0, -1.0]], dtype=DTYPE) # broadcasting
ex020_grid_before_op = torch.tensor([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], dtype=DTYPE)
ex020_out_shape = (2, 3)
ex020_out = torch.tensor([[2.0, 2.0, 2.0], [-1.0, -1.0, -1.0]], dtype=DTYPE)

In [61]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex020_rows_shape", "ex020", "rows_shape")
_check_private_value("ex020_grid_shape", "ex020", "grid_shape")
_check_private_value("ex020_rows_col_shape", "ex020", "rows_col_shape")
_check_private_tensor("ex020_rows_col", "ex020", "rows_col")
_check_private_tensor("ex020_rows_col_before_op", "ex020", "rows_col_before_op")
_check_private_tensor("ex020_grid_before_op", "ex020", "grid_before_op")
_check_private_value("ex020_out_shape", "ex020", "out_shape")
_check_private_tensor("ex020_out", "ex020", "out")


PASS: ex020_rows_shape
PASS: ex020_grid_shape
PASS: ex020_rows_col_shape
PASS: ex020_rows_col
PASS: ex020_rows_col_before_op
PASS: ex020_grid_before_op
PASS: ex020_out_shape
PASS: ex020_out


### Exercise 021 — Prepare column-owned values

**Purpose:** Add a size-one axis at the beginning so the vector clearly means one value per column.

**Visible inputs:** `cols`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `cols_row + grid`

**Task:** Create `cols_row` with `cols[None, :]`, then add it across rows.

**Ingredients:** The inserted leading axis is real shape metadata; the following addition performs broadcasting.

**Axis meaning in this exercise:** Axis `0` of `cols` means matrix columns. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `None` means insert a new axis of size `1`, while `:` keeps every existing value. Therefore `cols[None, :]` adds one row position before the column values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Unsqueeze before an axis


In [62]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
cols = torch.tensor([1.0, 10.0, 100.0], dtype=DTYPE)
grid = torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=DTYPE)
_register_case("ex021", {'cols': cols, 'grid': grid})


In [63]:
# Exercise 021: complete only the fields requested above; do not use autograd.
# Define `ex021_cols_shape`.
# Define `ex021_grid_shape`.
# Define `ex021_cols_row_shape`.
# Define `ex021_cols_row` with PyTorch tensor operations.
# Define `ex021_cols_row_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex021_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex021_out_shape`.
# Define `ex021_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex021_cols_shape = (3,)
ex021_grid_shape = (2, 3)
ex021_cols_row_shape = (1, 3)
ex021_cols_row = cols[None, :]
ex021_cols_row_before_op = torch.tensor([[1.0, 10.0, 100.0], [1.0, 10.0, 100.0]], dtype=DTYPE) # broadcasting
ex021_grid_before_op = torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=DTYPE)
ex021_out_shape = (2, 3)
ex021_out = torch.tensor([[1.0, 10.0, 100.0], [1.0, 10.0, 100.0]], dtype=DTYPE)

In [64]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex021_cols_shape", "ex021", "cols_shape")
_check_private_value("ex021_grid_shape", "ex021", "grid_shape")
_check_private_value("ex021_cols_row_shape", "ex021", "cols_row_shape")
_check_private_tensor("ex021_cols_row", "ex021", "cols_row")
_check_private_tensor("ex021_cols_row_before_op", "ex021", "cols_row_before_op")
_check_private_tensor("ex021_grid_before_op", "ex021", "grid_before_op")
_check_private_value("ex021_out_shape", "ex021", "out_shape")
_check_private_tensor("ex021_out", "ex021", "out")


PASS: ex021_cols_shape
PASS: ex021_grid_shape
PASS: ex021_cols_row_shape
PASS: ex021_cols_row
PASS: ex021_cols_row_before_op
PASS: ex021_grid_before_op
PASS: ex021_out_shape
PASS: ex021_out


### Exercise 022 — Unsqueeze before an axis

**Purpose:** Learn that `unsqueeze(0)` adds the same leading size-one axis as `values[None, :]`.

**Visible inputs:** `values`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `prepared + grid`

**Task:** Create `prepared = values.unsqueeze(0)`, then add it to the grid.

**Ingredients:** `unsqueeze(0)` inserts a new axis before the original vector axis.

**Axis meaning in this exercise:** Axis `0` of `values` means columns. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `unsqueeze(0)` inserts a new size-one axis at position `0`, before the vector axis. It changes shape information without copying tensor values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Unsqueeze after an axis


In [65]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
values = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
grid = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]], dtype=DTYPE)
_register_case("ex022", {'values': values, 'grid': grid})


In [66]:
# Exercise 022: complete only the fields requested above; do not use autograd.
# Define `ex022_values_shape`.
# Define `ex022_grid_shape`.
# Define `ex022_prepared_shape`.
# Define `ex022_prepared` with PyTorch tensor operations.
# Define `ex022_prepared_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex022_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex022_out_shape`.
# Define `ex022_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex022_values_shape = (3,)
ex022_grid_shape = (2, 3)
ex022_prepared_shape = (1, 3)
ex022_prepared = values.unsqueeze(0) # values[None, :]
ex022_prepared_before_op = torch.tensor([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]], dtype=DTYPE)
ex022_grid_before_op = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]], dtype=DTYPE)
ex022_out_shape = (2, 3)
ex022_out = torch.tensor([[11.0, 22.0, 33.0], [41.0, 52.0, 63.0]], dtype=DTYPE)

In [67]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex022_values_shape", "ex022", "values_shape")
_check_private_value("ex022_grid_shape", "ex022", "grid_shape")
_check_private_value("ex022_prepared_shape", "ex022", "prepared_shape")
_check_private_tensor("ex022_prepared", "ex022", "prepared")
_check_private_tensor("ex022_prepared_before_op", "ex022", "prepared_before_op")
_check_private_tensor("ex022_grid_before_op", "ex022", "grid_before_op")
_check_private_value("ex022_out_shape", "ex022", "out_shape")
_check_private_tensor("ex022_out", "ex022", "out")


PASS: ex022_values_shape
PASS: ex022_grid_shape
PASS: ex022_prepared_shape
PASS: ex022_prepared
PASS: ex022_prepared_before_op
PASS: ex022_grid_before_op
PASS: ex022_out_shape
PASS: ex022_out


### Exercise 023 — Unsqueeze after an axis

**Purpose:** Use `unsqueeze(1)` to turn row-owned values into a column.

**Visible inputs:** `values`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `prepared * grid`

**Task:** Create `prepared = values.unsqueeze(1)`, then multiply across columns.

**Ingredients:** Axis `1` is inserted after the existing axis `0`.

**Axis meaning in this exercise:** Axis `0` of `values` means rows. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `unsqueeze(1)` inserts a new size-one axis at position `1`, after the vector axis. It changes shape information without copying tensor values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Prepare image-channel values


In [68]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
values = torch.tensor([2.0, 3.0], dtype=DTYPE)
grid = torch.tensor([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], dtype=DTYPE)
_register_case("ex023", {'values': values, 'grid': grid})


In [69]:
# Exercise 023: complete only the fields requested above; do not use autograd.
# Define `ex023_values_shape`.
# Define `ex023_grid_shape`.
# Define `ex023_prepared_shape`.
# Define `ex023_prepared` with PyTorch tensor operations.
# Define `ex023_prepared_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex023_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex023_out_shape`.
# Define `ex023_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex023_values_shape = (2,)
ex023_grid_shape = (2, 3)
ex023_prepared_shape = (2, 1)
ex023_prepared = values.unsqueeze(dim=1)  # values[:, None]
ex023_prepared_before_op = torch.tensor([[2.0, 2.0, 2.0], [3.0, 3.0, 3.0]], dtype=DTYPE)
ex023_grid_before_op = grid
ex023_out_shape = (2, 3)
ex023_out = torch.tensor([[2.0, 2.0, 2.0], [3.0, 3.0, 3.0]], dtype=DTYPE)

In [70]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex023_values_shape", "ex023", "values_shape")
_check_private_value("ex023_grid_shape", "ex023", "grid_shape")
_check_private_value("ex023_prepared_shape", "ex023", "prepared_shape")
_check_private_tensor("ex023_prepared", "ex023", "prepared")
_check_private_tensor("ex023_prepared_before_op", "ex023", "prepared_before_op")
_check_private_tensor("ex023_grid_before_op", "ex023", "grid_before_op")
_check_private_value("ex023_out_shape", "ex023", "out_shape")
_check_private_tensor("ex023_out", "ex023", "out")


PASS: ex023_values_shape
PASS: ex023_grid_shape
PASS: ex023_prepared_shape
PASS: ex023_prepared
PASS: ex023_prepared_before_op
PASS: ex023_grid_before_op
PASS: ex023_out_shape
PASS: ex023_out


### Exercise 024 — Prepare image-channel values

**Purpose:** Place a channel vector on the channel axis of `(batch, channel, height, width)`.

**Visible inputs:** `channels`, `images`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `channel_view + images`

**Task:** First write `channel_view` as an explicit tensor literal with shape `(1, 3, 1, 1)`. It must show exactly how `[0.1, 0.2, 0.3]` is arranged by `channels[None, :, None, None]`. Then expand that prepared tensor logically to the common `(2, 3, 2, 2)` shape before adding it to every image location.

**Ingredients:** Follow the visible shape transformation `(3,)` → `(1, 3, 1, 1)` → `(2, 3, 2, 2)`. The four final axes are `(batch, channel, height, width)`; only the channel axis keeps size three during preparation.

**Axis meaning in this exercise:** `channels` uses axis `0` for channels. `images` uses `(batch, channel, height, width)`.

**New operation or idea explained:** Each `None` inside brackets inserts a size-one axis. `channels[None, :, None, None]` keeps the three channel values on axis `1` and creates size-one batch, height, and width axes. The literal `ex024_channel_view` must make this intermediate `(1, 3, 1, 1)` arrangement visible before `ex024_channel_view_before_op` shows its expansion to `(2, 3, 2, 2)`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Large-tensor encouragement:** Each complete operator-ready tensor in this exercise contains `24` values, so this exercise is intentionally slower and heavier than most. Work one axis and one repeated block at a time, using indentation to keep the axes visible. It is okay to use AI to help format or check the nested `torch.tensor(...)` literals, as long as you can explain how the original shapes reach the common shape and why each repeated block appears.

**Right-to-left expansion tip:** Build `ex024_channel_view_before_op` in stages rather than imagining all `24` values at once:

1. Expand width: `(1, 3, 1, 1)` → `(1, 3, 1, 2)` by repeating each channel value inside its innermost list.
2. Expand height: `(1, 3, 1, 2)` → `(1, 3, 2, 2)` by repeating each completed width row.
3. Leave the channel axis unchanged because its size is already `3`.
4. Expand batch: `(1, 3, 2, 2)` → `(2, 3, 2, 2)` by repeating the complete three-channel block.

Use one indentation level per axis so each kind of repetition remains visible.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Align a vector with a matrix


In [71]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
channels = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
images = torch.zeros((2, 3, 2, 2), dtype=DTYPE)
_register_case("ex024", {'channels': channels, 'images': images})


In [72]:
# Exercise 024: complete only the fields requested above; do not use autograd.
# Define `ex024_channels_shape`.
# Define `ex024_images_shape`.
# Define `ex024_channel_view_shape`.
# Define `ex024_channel_view` as an explicit `torch.tensor(...)` literal.
# Define `ex024_channel_view_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex024_images_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex024_out_shape`.
# Define `ex024_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex024_channels_shape = (3,)
ex024_images_shape = (2, 3, 2, 2)
ex024_channel_view_shape = (1, 3, 1, 1)
ex024_channel_view = torch.tensor(
    [
        [
            [
                [0.1],
            ],
            [
                [0.2],
            ],
            [
                [0.3],
            ],
        ],
    ],
    dtype=DTYPE,
)
ex024_channel_view_before_op = torch.tensor(
    [
        [
            [
                [0.1, 0.1],
                [0.1, 0.1],
            ],
            [
                [0.2, 0.2],
                [0.2, 0.2],
            ],
            [
                [0.3, 0.3],
                [0.3, 0.3],
            ],
        ],
        [
            [
                [0.1, 0.1],
                [0.1, 0.1],
            ],
            [
                [0.2, 0.2],
                [0.2, 0.2],
            ],
            [
                [0.3, 0.3],
                [0.3, 0.3],
            ],
        ],
    ],
    dtype=DTYPE,
)
ex024_images_before_op = torch.tensor(
    [
        [
            [
                [0.0, 0.0],
                [0.0, 0.0],
            ],
            [
                [0.0, 0.0],
                [0.0, 0.0],
            ],
            [
                [0.0, 0.0],
                [0.0, 0.0],
            ],
        ],
        [
            [
                [0.0, 0.0],
                [0.0, 0.0],
            ],
            [
                [0.0, 0.0],
                [0.0, 0.0],
            ],
            [
                [0.0, 0.0],
                [0.0, 0.0],
            ],
        ],
    ],
    dtype=DTYPE,
)
ex024_out_shape = (2, 3, 2, 2)
ex024_out = torch.tensor(
    [
        [
            [
                [0.1, 0.1],
                [0.1, 0.1],
            ],
            [
                [0.2, 0.2],
                [0.2, 0.2],
            ],
            [
                [0.3, 0.3],
                [0.3, 0.3],
            ],
        ],
        [
            [
                [0.1, 0.1],
                [0.1, 0.1],
            ],
            [
                [0.2, 0.2],
                [0.2, 0.2],
            ],
            [
                [0.3, 0.3],
                [0.3, 0.3],
            ],
        ],
    ],
    dtype=DTYPE,
)


In [73]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex024_channels_shape", "ex024", "channels_shape")
_check_private_value("ex024_images_shape", "ex024", "images_shape")
_check_private_value("ex024_channel_view_shape", "ex024", "channel_view_shape")
_check_private_tensor("ex024_channel_view", "ex024", "channel_view")
_check_private_tensor("ex024_channel_view_before_op", "ex024", "channel_view_before_op")
_check_private_tensor("ex024_images_before_op", "ex024", "images_before_op")
_check_private_value("ex024_out_shape", "ex024", "out_shape")
_check_private_tensor("ex024_out", "ex024", "out")


PASS: ex024_channels_shape
PASS: ex024_images_shape
PASS: ex024_channel_view_shape
PASS: ex024_channel_view
PASS: ex024_channel_view_before_op
PASS: ex024_images_before_op
PASS: ex024_out_shape
PASS: ex024_out


## 4. Alignment as a dedicated reasoning tool

Alignment is a paper method for answering one question:

> **Which axis of one operand is compared with which axis of the other operand?**

It does not create a tensor, copy values, or say that anything expands.

This section covers Exercises 025–027.

### Is mental alignment necessary?

Yes, when shapes differ or an operation fails. It is the most reliable way to debug broadcasting. You do **not** need to write aligned shapes for every obvious operation forever. These exercises isolate the skill until the procedure becomes available when needed.

### The alignment procedure

Suppose a matrix has actual shape `(3, 5)` and a vector has actual shape `(5,)`.

1. Count axes. The matrix has two axes; the vector has one.
2. Place the final dimensions under each other.
3. For comparison only, fill missing positions on the **left** with `1`.

```text
actual matrix shape:  (3, 5)
actual vector shape:     (5,)

aligned matrix shape: (3, 5)
aligned vector shape: (1, 5)
axis numbers:          0  1
```

The vector's real shape is still `(5,)`. Writing `(1, 5)` does not call `reshape`; it only records where the vector axis sits during comparison.

Now compare one aligned axis at a time:

- axis `1`: sizes `5` and `5` are equal;
- axis `0`: sizes `3` and `1` are compatible because one size is `1`.

The result uses the non-one size at each compatible axis, so this example's result shape is `(3, 5)`.

A scalar has actual shape `()`. When compared with a rank-two tensor, its conceptual aligned shape has two size-one positions. Exercises 025–027 move from equal shapes to scalars, vectors, matrices, and rank-three tensors one step at a time.

**Alignment asks where axes line up. It does not yet ask where values are reused.**


### Exercise 025 — Align a vector with a matrix

**Purpose:** See why a vector lines up with the final matrix axis.

**Visible inputs:** `matrix`, `vector`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + vector`

**Task:** Write both actual shapes, then left-pad only the shorter shape for comparison.

**Ingredients:** Alignment is bookkeeping about axis positions. It does not yet ask which axis expands.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`; the vector values belong to columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Subtract a background from an image batch


In [74]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
vector = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex025", {'matrix': matrix, 'vector': vector})


In [75]:
# Exercise 025: complete only the fields requested above; do not use autograd.
# Define `ex025_matrix_shape`.
# Define `ex025_matrix_aligned_shape`.
# Define `ex025_vector_shape`.
# Define `ex025_vector_aligned_shape`.
# Define `ex025_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex025_vector_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex025_out_shape`.
# Define `ex025_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex025_matrix_shape = (2, 3)
ex025_matrix_aligned_shape = (2, 3)
ex025_vector_shape = (3,)
ex025_vector_aligned_shape = (1, 3)
ex025_matrix_before_op = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
ex025_vector_before_op = torch.tensor([[10.0, 20.0, 30.0], [10.0, 20.0, 30.0]], dtype=DTYPE)
ex025_out_shape = (2, 3)
ex025_out = torch.tensor([[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]], dtype=DTYPE)

In [76]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex025_matrix_shape", "ex025", "matrix_shape")
_check_private_value("ex025_matrix_aligned_shape", "ex025", "matrix_aligned_shape")
_check_private_value("ex025_vector_shape", "ex025", "vector_shape")
_check_private_value("ex025_vector_aligned_shape", "ex025", "vector_aligned_shape")
_check_private_tensor("ex025_matrix_before_op", "ex025", "matrix_before_op")
_check_private_tensor("ex025_vector_before_op", "ex025", "vector_before_op")
_check_private_value("ex025_out_shape", "ex025", "out_shape")
_check_private_tensor("ex025_out", "ex025", "out")


PASS: ex025_matrix_shape
PASS: ex025_matrix_aligned_shape
PASS: ex025_vector_shape
PASS: ex025_vector_aligned_shape
PASS: ex025_matrix_before_op
PASS: ex025_vector_before_op
PASS: ex025_out_shape
PASS: ex025_out


### Exercise 026 — Subtract a background from an image batch

**Purpose:** Subtract the same grayscale background from every image in a batch. This shows how a `(height, width)` tensor is reused when the larger tensor has an additional batch axis.

**Visible inputs:** `images`, `background`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `images - background`

**Task:** Write the actual and aligned shapes, then subtract the background from every image.

**Ingredients:** `images` has shape `(batch, height, width)`. `background` has shape `(height, width)` and no batch axis. These are grayscale images, so there is no channel axis.

**Axis meaning in this exercise:** `images` uses `(batch, height, width)`; `background` uses `(height, width)`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Large-tensor encouragement:** Each complete operator-ready tensor in this exercise contains `24` values, so this exercise is intentionally slower and heavier than most. Work one axis and one repeated block at a time, using indentation to keep the axes visible. It is okay to use AI to help format or check the nested `torch.tensor(...)` literals, as long as you can explain how the original shapes reach the common shape and why each repeated block appears.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Complete alignment check


In [77]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
images = torch.tensor(
    [
        [
            [0.0, 1.0, 2.0, 3.0],
            [4.0, 5.0, 6.0, 7.0],
            [8.0, 9.0, 10.0, 11.0],
        ],
        [
            [12.0, 13.0, 14.0, 15.0],
            [16.0, 17.0, 18.0, 19.0],
            [20.0, 21.0, 22.0, 23.0],
        ],
    ],
    dtype=DTYPE,
)
background = torch.tensor(
    [
        [1.0, 2.0, 3.0, 4.0],
        [5.0, 6.0, 7.0, 8.0],
        [9.0, 10.0, 11.0, 12.0],
    ],
    dtype=DTYPE,
)
_register_case("ex026", {'images': images, 'background': background})


In [78]:
# Exercise 026: complete only the fields requested above; do not use autograd.
# Define `ex026_images_shape`.
# Define `ex026_images_aligned_shape`.
# Define `ex026_background_shape`.
# Define `ex026_background_aligned_shape`.
# Define `ex026_images_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex026_background_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex026_out_shape`.
# Define `ex026_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex026_images_shape = (2, 3, 4)
ex026_images_aligned_shape = (2, 3, 4)
ex026_background_shape = (3, 4)
ex026_background_aligned_shape = (1, 3, 4)
ex026_images_before_op = images
ex026_background_before_op = torch.tensor(
    [
        [
            [1.0, 2.0, 3.0, 4.0],
            [5.0, 6.0, 7.0, 8.0],
            [9.0, 10.0, 11.0, 12.0],
        ],
        [
            [1.0, 2.0, 3.0, 4.0],
            [5.0, 6.0, 7.0, 8.0],
            [9.0, 10.0, 11.0, 12.0],
        ],
    ],
    dtype=DTYPE,
)
ex026_out_shape = (2, 3, 4)
ex026_out= torch.tensor(
    [
        [
            [-1.0, -1.0, -1.0, -1.0],
            [-1.0, -1.0, -1.0, -1.0],
            [-1.0, -1.0, -1.0, -1.0],
        ],
        [
            [11.0, 11.0, 11.0, 11.0],
            [11.0, 11.0, 11.0, 11.0],
            [11.0, 11.0, 11.0, 11.0],
        ],
    ],
    dtype=DTYPE,
)

In [79]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex026_images_shape", "ex026", "images_shape")
_check_private_value("ex026_images_aligned_shape", "ex026", "images_aligned_shape")
_check_private_value("ex026_background_shape", "ex026", "background_shape")
_check_private_value("ex026_background_aligned_shape", "ex026", "background_aligned_shape")
_check_private_tensor("ex026_images_before_op", "ex026", "images_before_op")
_check_private_tensor("ex026_background_before_op", "ex026", "background_before_op")
_check_private_value("ex026_out_shape", "ex026", "out_shape")
_check_private_tensor("ex026_out", "ex026", "out")


PASS: ex026_images_shape
PASS: ex026_images_aligned_shape
PASS: ex026_background_shape
PASS: ex026_background_aligned_shape
PASS: ex026_images_before_op
PASS: ex026_background_before_op
PASS: ex026_out_shape
PASS: ex026_out


### Exercise 027 — Complete alignment check

**Purpose:** Practice the complete alignment procedure before using it to diagnose invalid shapes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Right-align both operands, determine the result shape, then add.

**Ingredients:** Work from the final axis leftward. Equal sizes or a size `1` are compatible.

**Axis meaning in this exercise:** Treat the three result axes as `(batch, rows, columns)`; `b` supplies column values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Prepare per-image width offsets


In [80]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[[1.0], [2.0], [3.0]]], dtype=DTYPE)
b = torch.tensor([10.0, 20.0, 30.0, 40.0], dtype=DTYPE)
_register_case("ex027", {'a': a, 'b': b})


In [87]:
# Exercise 027: complete only the fields requested above; do not use autograd.
# Define `ex027_a_shape`.
# Define `ex027_a_aligned_shape`.
# Define `ex027_b_shape`.
# Define `ex027_b_aligned_shape`.
# Define `ex027_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex027_b_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex027_out_shape`.
# Define `ex027_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex027_a_shape = (1, 3, 1)
ex027_b_shape = (4,)
ex027_a_aligned_shape = (1, 3, 1)
ex027_b_aligned_shape = (1, 1, 4)
ex027_out_shape = (1, 3, 4)
ex027_a_before_op = torch.tensor(
    [
        [
            [1.0, 1.0, 1.0, 1.0],
            [2.0, 2.0, 2.0, 2.0],
            [3.0, 3.0, 3.0, 3.0],
        ]
    ],
    dtype=DTYPE
)
ex027_b_before_op = torch.tensor(
    [
        [
            [10.0, 20.0, 30.0, 40.0],
            [10.0, 20.0, 30.0, 40.0],
            [10.0, 20.0, 30.0, 40.0],
        ]
    ],
    dtype=DTYPE
)
ex027_out = torch.tensor(
    [
        [
            [11.0, 21.0, 31.0, 41.0],
            [12.0, 22.0, 32.0, 42.0],
            [13.0, 23.0, 33.0, 43.0],
        ]
    ],
    dtype=DTYPE
)

In [88]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex027_a_shape", "ex027", "a_shape")
_check_private_value("ex027_a_aligned_shape", "ex027", "a_aligned_shape")
_check_private_value("ex027_b_shape", "ex027", "b_shape")
_check_private_value("ex027_b_aligned_shape", "ex027", "b_aligned_shape")
_check_private_tensor("ex027_a_before_op", "ex027", "a_before_op")
_check_private_tensor("ex027_b_before_op", "ex027", "b_before_op")
_check_private_value("ex027_out_shape", "ex027", "out_shape")
_check_private_tensor("ex027_out", "ex027", "out")


PASS: ex027_a_shape
PASS: ex027_a_aligned_shape
PASS: ex027_b_shape
PASS: ex027_b_aligned_shape
PASS: ex027_a_before_op
PASS: ex027_b_before_op
PASS: ex027_out_shape
PASS: ex027_out


## 5. Shape preparation and multi-operand broadcasting

These exercises end with real tensor computations rather than shape diagnosis alone.

Exercise 028 inserts a missing size-one axis, writes the complete operator-ready tensors, and adds them. Exercise 029 makes three differently shaped operands reach one common shape before `torch.where` selects the output values.

This section covers Exercises 028–029.


### Exercise 028 — Prepare per-image width offsets

**Purpose:** Give each grayscale image its own width-specific offsets and reuse those offsets down the image height. A `(batch, width)` tensor cannot express that directly, so you will insert a size-one height axis and then perform the addition.

**Visible inputs:** `images`, `per_image_width_offset`. Run the fixture below and read its explicit values; it prints no expected answers.

**Operation to reason about:** `prepared + images`

**Task:** Create `prepared = per_image_width_offset[:, None, :]`, write both complete operator-ready tensors, then calculate the output.

**Ingredients:** The offset shape changes from `(batch, width)` to `(batch, 1, width)`, then broadcasts with `(batch, height, width)`.

**Axis meaning in this exercise:** `images` uses `(batch, height, width)`. `per_image_width_offset` uses `(batch, width)`, and the inserted middle axis represents height. These are grayscale images, so there is no channel axis.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator.
- **Singleton axis:** an axis with size `1`; its values may be reused along that direction.

**Explicit operator-ready tensors:** Write each requested operand at the common `(batch, height, width)` shape with a literal `torch.tensor(...)`. This makes the height-wise reuse visible before addition.

**Order:** Predict shapes first, then write the prepared and operator-ready tensors, then calculate the output.

**Next concept:** Broadcast three operands with `torch.where`


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
images = torch.tensor(
    [
        [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]],
        [[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]],
    ],
    dtype=DTYPE,
)
per_image_width_offset = torch.tensor(
    [[10.0, 20.0, 30.0], [100.0, 200.0, 300.0]],
    dtype=DTYPE,
)
_register_case("ex028", {'images': images, 'per_image_width_offset': per_image_width_offset})


In [ ]:
# Exercise 028: complete only the fields requested above; do not use autograd.
# Define `ex028_images_shape`.
# Define `ex028_per_image_width_offset_shape`.
# Define `ex028_prepared_shape`.
# Define `ex028_prepared` with `per_image_width_offset[:, None, :]`.
# Define `ex028_prepared_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex028_images_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex028_out_shape`.
# Define `ex028_out` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex028_images_shape", "ex028", "images_shape")
_check_private_value("ex028_per_image_width_offset_shape", "ex028", "per_image_width_offset_shape")
_check_private_value("ex028_prepared_shape", "ex028", "prepared_shape")
_check_private_tensor("ex028_prepared", "ex028", "prepared")
_check_private_tensor("ex028_prepared_before_op", "ex028", "prepared_before_op")
_check_private_tensor("ex028_images_before_op", "ex028", "images_before_op")
_check_private_value("ex028_out_shape", "ex028", "out_shape")
_check_private_tensor("ex028_out", "ex028", "out")


### Exercise 029 — Broadcast three operands with `torch.where`

**Purpose:** Make three operands reach one common shape before `torch.where` selects values. The mask varies by row, `x` varies by column, and scalar `y` supplies one fallback value everywhere.

**Visible inputs:** `mask`, `x`, `y`. Run the fixture below and read its explicit values; it prints no expected answers.

**Operation to reason about:** `torch.where(mask, x, y)`

**Task:** Write the complete operator-ready mask, `x`, and `y` tensors at their common shape, then calculate the selected output.

**Ingredients:** `torch.where(mask, x, y)` chooses from `x` where the Boolean mask is `True` and from `y` where it is `False`. All three operands broadcast to `(2, 3)` first.

**Axis meaning in this exercise:** Result axis `0` represents rows and axis `1` represents columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operation.
- **Boolean mask:** a tensor of `True` and `False` values controlling which operand supplies each result value.

**Explicit operator-ready tensors:** Write all three requested common-shape operands with literal `torch.tensor(...)` values. Use `torch.bool` for the mask and `DTYPE` for `x`, `y`, and the output.

**Order:** Predict shapes first, then write all operator-ready tensors, then calculate the output.

**Next concept:** Batch feature bias


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
mask = torch.tensor([[True], [False]], dtype=torch.bool)
x = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
y = torch.tensor(-1.0, dtype=DTYPE)
_register_case("ex029", {'mask': mask, 'x': x, 'y': y})


In [ ]:
# Exercise 029: complete only the fields requested above; do not use autograd.
# Define `ex029_mask_shape`.
# Define `ex029_x_shape`.
# Define `ex029_y_shape`.
# Define `ex029_mask_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex029_x_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex029_y_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex029_out_shape`.
# Define `ex029_out` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex029_mask_shape", "ex029", "mask_shape")
_check_private_value("ex029_x_shape", "ex029", "x_shape")
_check_private_value("ex029_y_shape", "ex029", "y_shape")
_check_private_tensor("ex029_mask_before_op", "ex029", "mask_before_op")
_check_private_tensor("ex029_x_before_op", "ex029", "x_before_op")
_check_private_tensor("ex029_y_before_op", "ex029", "y_before_op")
_check_private_value("ex029_out_shape", "ex029", "out_shape")
_check_private_tensor("ex029_out", "ex029", "out")


## 6. Machine-learning axis patterns

Apply broadcasting to examples, features, tokens, image channels, attention axes, and pairwise items.

This section covers Exercises 030–037.


### Exercise 030 — Batch feature bias

**Purpose:** Recognize the standard `(examples, features) + (features,)` pattern.

**Visible inputs:** `activations`, `bias`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `activations + bias`

**Task:** Add one bias per feature to every example.

**Ingredients:** Axes are `(examples, features)`; the feature vector aligns last.

**Axis meaning in this exercise:** `activations` uses `(examples, features)`; the bias vector values belong to features.

**New operation or idea explained:** A **bias** is a value added to another value. Here there is one bias value for each feature, and the same feature biases are used for every example.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Per-example scalar offset


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
activations = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]], dtype=DTYPE)
bias = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
_register_case("ex030", {'activations': activations, 'bias': bias})


In [ ]:
# Exercise 030: complete only the fields requested above; do not use autograd.
# Define `ex030_activations_shape`.
# Define `ex030_bias_shape`.
# Define `ex030_out_shape`.
# Define `ex030_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex030_activations_shape", "ex030", "activations_shape")
_check_private_value("ex030_bias_shape", "ex030", "bias_shape")
_check_private_value("ex030_out_shape", "ex030", "out_shape")
_check_private_tensor("ex030_out", "ex030", "out")


### Exercise 031 — Per-example scalar offset

**Purpose:** Prepare one value for every feature in each example.

**Visible inputs:** `activations`, `offset`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `offset_view + activations`

**Task:** Create `offset_view = offset[:, None]`, then add it across features.

**Ingredients:** The prepared axes are `(examples, 1)`.

**Axis meaning in this exercise:** `activations` uses `(examples, features)`; offset axis `0` identifies examples.

**New operation or idea explained:** An **offset** is an added value. Here each example owns one offset that must be reused across all of that example’s features.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Position embeddings across batches


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
activations = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
offset = torch.tensor([0.5, -1.0], dtype=DTYPE)
_register_case("ex031", {'activations': activations, 'offset': offset})


In [ ]:
# Exercise 031: complete only the fields requested above; do not use autograd.
# Define `ex031_activations_shape`.
# Define `ex031_offset_shape`.
# Define `ex031_offset_view_shape`.
# Define `ex031_offset_view` with PyTorch tensor operations.
# Define `ex031_out_shape`.
# Define `ex031_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex031_activations_shape", "ex031", "activations_shape")
_check_private_value("ex031_offset_shape", "ex031", "offset_shape")
_check_private_value("ex031_offset_view_shape", "ex031", "offset_view_shape")
_check_private_tensor("ex031_offset_view", "ex031", "offset_view")
_check_private_value("ex031_out_shape", "ex031", "out_shape")
_check_private_tensor("ex031_out", "ex031", "out")


### Exercise 032 — Position embeddings across batches

**Purpose:** Reuse one `(time, features)` table for every batch item.

**Visible inputs:** `tokens`, `positions`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `tokens + positions`

**Task:** Add one position-feature row to each batch.

**Ingredients:** Axes are `(batch, time, features)` and `(time, features)`.

**Axis meaning in this exercise:** `tokens` uses `(batch, time, features)`; `positions` uses `(time, features)`.

**New operation or idea explained:** A **position embedding** is a row of feature values associated with one sequence position. The same position table is added to every batch item.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Token validity mask


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
tokens = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
positions = torch.tensor([[-0.5, 0.0, 0.5, 1.0], [0.0, 0.5, 1.0, -0.5], [0.5, 1.0, -0.5, 0.0]], dtype=DTYPE)
_register_case("ex032", {'tokens': tokens, 'positions': positions})


In [ ]:
# Exercise 032: complete only the fields requested above; do not use autograd.
# Define `ex032_tokens_shape`.
# Define `ex032_positions_shape`.
# Define `ex032_out_shape`.
# Define `ex032_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex032_tokens_shape", "ex032", "tokens_shape")
_check_private_value("ex032_positions_shape", "ex032", "positions_shape")
_check_private_value("ex032_out_shape", "ex032", "out_shape")
_check_private_tensor("ex032_out", "ex032", "out")


### Exercise 033 — Token validity mask

**Purpose:** Expand one validity value over all embedding features.

**Visible inputs:** `mask`, `embeddings`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `mask_view * embeddings`

**Task:** Create `mask_view = mask[:, :, None]`, then zero all features at invalid token positions.

**Ingredients:** Mask axes are `(batch, time)`; insert a singleton feature axis.

**Axis meaning in this exercise:** `mask` uses `(batch, time)`; `embeddings` uses `(batch, time, features)`.

**New operation or idea explained:** A **mask** marks which positions to keep. Here `1.0` means keep the embedding and `0.0` means replace it with zeros through multiplication. An **embedding** is the feature vector representing one token.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Large-tensor encouragement:** Each complete operator-ready tensor in this exercise contains `24` values, so this exercise is intentionally slower and heavier than most. Work one axis and one repeated block at a time, using indentation to keep the axes visible. It is okay to use AI to help format or check the nested `torch.tensor(...)` literals, as long as you can explain how the original shapes reach the common shape and why each repeated block appears.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Class weights


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
mask = torch.tensor([[1.0, 1.0, 0.0], [1.0, 0.0, 0.0]], dtype=DTYPE)
embeddings = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex033", {'mask': mask, 'embeddings': embeddings})


In [ ]:
# Exercise 033: complete only the fields requested above; do not use autograd.
# Define `ex033_mask_shape`.
# Define `ex033_embeddings_shape`.
# Define `ex033_mask_view_shape`.
# Define `ex033_mask_view` with PyTorch tensor operations.
# Define `ex033_mask_view_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex033_embeddings_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex033_out_shape`.
# Define `ex033_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex033_mask_shape", "ex033", "mask_shape")
_check_private_value("ex033_embeddings_shape", "ex033", "embeddings_shape")
_check_private_value("ex033_mask_view_shape", "ex033", "mask_view_shape")
_check_private_tensor("ex033_mask_view", "ex033", "mask_view")
_check_private_tensor("ex033_mask_view_before_op", "ex033", "mask_view_before_op")
_check_private_tensor("ex033_embeddings_before_op", "ex033", "embeddings_before_op")
_check_private_value("ex033_out_shape", "ex033", "out_shape")
_check_private_tensor("ex033_out", "ex033", "out")


### Exercise 034 — Class weights

**Purpose:** Apply one weight per candidate class to every example.

**Visible inputs:** `losses`, `class_weights`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `losses * class_weights`

**Task:** Weight every candidate-class loss.

**Ingredients:** Axes are `(examples, classes)` and `(classes,)`. This weights all candidate losses; no target indexing occurs here.

**Axis meaning in this exercise:** `losses` uses `(examples, classes)`; the weight vector values belong to classes.

**New operation or idea explained:** A **loss** is a number measuring error. A **class** is one candidate category. This exercise multiplies every candidate-class loss by the weight assigned to that class; it does not choose a target class.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Image channel bias


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
losses = torch.tensor([[0.2, 0.5, 1.0], [0.3, 0.4, 0.8]], dtype=DTYPE)
class_weights = torch.tensor([1.0, 0.5, 2.0], dtype=DTYPE)
_register_case("ex034", {'losses': losses, 'class_weights': class_weights})


In [ ]:
# Exercise 034: complete only the fields requested above; do not use autograd.
# Define `ex034_losses_shape`.
# Define `ex034_class_weights_shape`.
# Define `ex034_out_shape`.
# Define `ex034_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex034_losses_shape", "ex034", "losses_shape")
_check_private_value("ex034_class_weights_shape", "ex034", "class_weights_shape")
_check_private_value("ex034_out_shape", "ex034", "out_shape")
_check_private_tensor("ex034_out", "ex034", "out")


### Exercise 035 — Image channel bias

**Purpose:** Prepare one added value for each channel in images shaped `(batch, channel, height, width)`.

**Visible inputs:** `images`, `bias`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `bias_view + images`

**Task:** Create `bias_view = bias[None, :, None, None]`, then add it to each image location.

**Ingredients:** Axes are `(batch, channel, height, width)`.

**Axis meaning in this exercise:** `images` uses `(batch, channel, height, width)`; the bias vector values belong to channels.

**New operation or idea explained:** A **channel bias** is one added value for each image channel. The same channel value is reused for every batch item, height position, and width position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Attention mask across batches and heads


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
images = torch.arange(48, dtype=DTYPE).reshape(2, 3, 2, 4)
bias = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
_register_case("ex035", {'images': images, 'bias': bias})


In [ ]:
# Exercise 035: complete only the fields requested above; do not use autograd.
# Define `ex035_images_shape`.
# Define `ex035_bias_shape`.
# Define `ex035_bias_view_shape`.
# Define `ex035_bias_view` with PyTorch tensor operations.
# Define `ex035_out_shape`.
# Define `ex035_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex035_images_shape", "ex035", "images_shape")
_check_private_value("ex035_bias_shape", "ex035", "bias_shape")
_check_private_value("ex035_bias_view_shape", "ex035", "bias_view_shape")
_check_private_tensor("ex035_bias_view", "ex035", "bias_view")
_check_private_value("ex035_out_shape", "ex035", "out_shape")
_check_private_tensor("ex035_out", "ex035", "out")


### Exercise 036 — Attention mask across batches and heads

**Purpose:** Use one query-by-key mask for every batch item and every attention head.

**Visible inputs:** `scores`, `mask`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `scores.masked_fill(~mask, -1e9)`

**Task:** Mask future key positions in every batch and head.

**Ingredients:** Score axes are `(batch, heads, query, key)`; mask axes are `(query, key)`.

**Axis meaning in this exercise:** `scores` uses `(batch, heads, query, key)`; `mask` uses `(query, key)`.

**New operation or idea explained:** An **attention score** measures how strongly one query position relates to one key position. The Boolean mask says which query-key pairs are allowed. `~mask` flips `True` and `False`; `masked_fill(condition, value)` replaces positions where `condition` is `True`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Large-tensor encouragement:** Each complete operator-ready tensor in this exercise contains `18` values, so this exercise is intentionally slower and heavier than most. Work one axis and one repeated block at a time, using indentation to keep the axes visible. It is okay to use AI to help format or check the nested `torch.tensor(...)` literals, as long as you can explain how the original shapes reach the common shape and why each repeated block appears.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Pairwise feature differences


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
scores = torch.arange(18, dtype=DTYPE).reshape(1, 2, 3, 3)
mask = torch.tensor([[True, False, False], [True, True, False], [True, True, True]], dtype=torch.bool)
_register_case("ex036", {'scores': scores, 'mask': mask})


In [ ]:
# Exercise 036: complete only the fields requested above; do not use autograd.
# Define `ex036_scores_shape`.
# Define `ex036_mask_shape`.
# Define `ex036_scores_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex036_inverted_mask_before_op` as an explicit Boolean `torch.tensor(...)` literal for `~mask`.
# Define `ex036_out_shape`.
# Define `ex036_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex036_scores_shape", "ex036", "scores_shape")
_check_private_value("ex036_mask_shape", "ex036", "mask_shape")
_check_private_tensor("ex036_scores_before_op", "ex036", "scores_before_op")
_check_private_tensor("ex036_inverted_mask_before_op", "ex036", "inverted_mask_before_op")
_check_private_value("ex036_out_shape", "ex036", "out_shape")
_check_private_tensor("ex036_out", "ex036", "out")


### Exercise 037 — Pairwise feature differences

**Purpose:** Add size-one item axes so every vector in `x` can be compared with every vector in `y`.

**Visible inputs:** `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `x_rows - y_rows`

**Task:** Create `x_rows = x[:, None, :]` and `y_rows = y[None, :, :]`, then compute every feature-wise pair difference.

**Ingredients:** Prepared axes are `(x_items, 1, features)` and `(1, y_items, features)`.

**Axis meaning in this exercise:** `x` uses `(x_items, features)` and `y` uses `(y_items, features)`.

**New operation or idea explained:** **Pairwise** means every item from `x` is compared with every item from `y`. A feature-wise difference keeps the feature axis instead of summing it yet.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Large-tensor encouragement:** Each complete operator-ready tensor in this exercise contains `18` values, so this exercise is intentionally slower and heavier than most. Work one axis and one repeated block at a time, using indentation to keep the axes visible. It is okay to use AI to help format or check the nested `torch.tensor(...)` literals, as long as you can explain how the original shapes reach the common shape and why each repeated block appears.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Expand a row explicitly


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
x = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
y = torch.tensor([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0], [3.0, 3.0, 3.0]], dtype=DTYPE)
_register_case("ex037", {'x': x, 'y': y})


In [ ]:
# Exercise 037: complete only the fields requested above; do not use autograd.
# Define `ex037_x_shape`.
# Define `ex037_y_shape`.
# Define `ex037_x_rows_shape`.
# Define `ex037_x_rows` with PyTorch tensor operations.
# Define `ex037_y_rows_shape`.
# Define `ex037_y_rows` with PyTorch tensor operations.
# Define `ex037_x_rows_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex037_y_rows_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex037_out_shape`.
# Define `ex037_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex037_x_shape", "ex037", "x_shape")
_check_private_value("ex037_y_shape", "ex037", "y_shape")
_check_private_value("ex037_x_rows_shape", "ex037", "x_rows_shape")
_check_private_tensor("ex037_x_rows", "ex037", "x_rows")
_check_private_value("ex037_y_rows_shape", "ex037", "y_rows_shape")
_check_private_tensor("ex037_y_rows", "ex037", "y_rows")
_check_private_tensor("ex037_x_rows_before_op", "ex037", "x_rows_before_op")
_check_private_tensor("ex037_y_rows_before_op", "ex037", "y_rows_before_op")
_check_private_value("ex037_out_shape", "ex037", "out_shape")
_check_private_tensor("ex037_out", "ex037", "out")


## 7. Explicit APIs, storage, and capstones

Use expand/repeat deliberately, then integrate broadcasting with reductions and realistic tensor programs.

This section covers Exercises 038–045.


### Exercise 038 — Expand a row explicitly

**Purpose:** Use `expand_as` to make a same-storage view that behaves like the target shape.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand the one-row source to the target shape.

**Ingredients:** `expand_as` may enlarge singleton dimensions; it does not accept incompatible non-singleton sizes.

**Axis meaning in this exercise:** For `source` and `target`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `source.expand_as(target)` returns a view of `source` that behaves as if it had `target.shape`. Only size-one axes may grow; repeated values are normally not copied.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Broadcast to a target shape


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex038", {'source': source, 'target': target})


In [ ]:
# Exercise 038: complete only the fields requested above; do not use autograd.
# Define `ex038_source_shape`.
# Define `ex038_target_shape`.
# Define `ex038_out_shape`.
# Define `ex038_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex038_source_shape", "ex038", "source_shape")
_check_private_value("ex038_target_shape", "ex038", "target_shape")
_check_private_value("ex038_out_shape", "ex038", "out_shape")
_check_private_tensor("ex038_out", "ex038", "out")


### Exercise 039 — Broadcast to a target shape

**Purpose:** Use `torch.broadcast_to` with the same rules used by elementwise operations.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `torch.broadcast_to(source, target.shape)`

**Task:** Broadcast the source to `target.shape`.

**Ingredients:** The target supplies only a shape; its uninitialized values are irrelevant.

**Axis meaning in this exercise:** The source vector axis means columns; `target` uses `(rows, columns)`.

**New operation or idea explained:** `torch.broadcast_to(source, target.shape)` returns a broadcasted view with the requested shape. `target` supplies only its shape; values created by `torch.empty` are intentionally uninitialized and must not be read.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Repeat materializes values


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex039", {'source': source, 'target': target})


In [ ]:
# Exercise 039: complete only the fields requested above; do not use autograd.
# Define `ex039_source_shape`.
# Define `ex039_target_shape`.
# Define `ex039_out_shape`.
# Define `ex039_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex039_source_shape", "ex039", "source_shape")
_check_private_value("ex039_target_shape", "ex039", "target_shape")
_check_private_value("ex039_out_shape", "ex039", "out_shape")
_check_private_tensor("ex039_out", "ex039", "out")


### Exercise 040 — Repeat materializes values

**Purpose:** Compare copied repetition with a broadcasted view that reuses stored values.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.repeat(2, 1)`

**Task:** Repeat the row twice.

**Ingredients:** `repeat(2, 1)` repeats axis `0` twice and axis `1` once.

**Axis meaning in this exercise:** For `source`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `repeat(2, 1)` physically repeats values: twice along axis `0` and once along axis `1`. Unlike broadcasting, `repeat` allocates repeated data.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Expanded view shares storage


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex040", {'source': source})


In [ ]:
# Exercise 040: complete only the fields requested above; do not use autograd.
# Define `ex040_source_shape`.
# Define `ex040_out_shape`.
# Define `ex040_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex040_source_shape", "ex040", "source_shape")
_check_private_value("ex040_out_shape", "ex040", "out_shape")
_check_private_tensor("ex040_out", "ex040", "out")


### Exercise 041 — Expanded view shares storage

**Purpose:** See that an expanded view and its source can refer to the same stored values.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.expand(2, 3)`

**Task:** Create the expanded view and predict whether it shares storage with `source`.

**Ingredients:** `expand` normally changes strides rather than copying values. This storage topic is optional for basic broadcasting use.

**Axis meaning in this exercise:** For `source`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `expand(2, 3)` returns a view that behaves like shape `(2, 3)` while reusing the source’s stored values. A **stride** controls movement through storage; stride zero rereads the same value.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.
- **Storage:** the memory holding tensor values. **Sharing storage** means two tensors refer to that same memory.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Clone materializes an expanded view


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex041", {'source': source})


In [ ]:
# Exercise 041: complete only the fields requested above; do not use autograd.
# Define `ex041_source_shape`.
# Define `ex041_shares_storage`.
# Define `ex041_out_shape`.
# Define `ex041_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex041_source_shape", "ex041", "source_shape")
_check_private_value("ex041_out_shape", "ex041", "out_shape")
_check_private_value("ex041_shares_storage", "ex041", "shares_storage")
_check_private_tensor("ex041_out", "ex041", "out")


### Exercise 042 — Clone materializes an expanded view

**Purpose:** Turn a broadcasted view into independent storage.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.expand(2, 3).clone()`

**Task:** Expand, clone, and predict storage sharing.

**Ingredients:** Cloning allocates independent storage for the visible values.

**Axis meaning in this exercise:** For `source`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `clone()` creates a new tensor with its own stored values. After `expand(...).clone()`, the visible repeated values no longer share storage with the source.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.
- **Storage:** the memory holding tensor values. **Sharing storage** means two tensors refer to that same memory.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Final practice: column normalization


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex042", {'source': source})


In [ ]:
# Exercise 042: complete only the fields requested above; do not use autograd.
# Define `ex042_source_shape`.
# Define `ex042_shares_storage`.
# Define `ex042_out_shape`.
# Define `ex042_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex042_source_shape", "ex042", "source_shape")
_check_private_value("ex042_out_shape", "ex042", "out_shape")
_check_private_value("ex042_shares_storage", "ex042", "shares_storage")
_check_private_tensor("ex042_out", "ex042", "out")


### Exercise 043 — Final practice: column normalization

**Purpose:** Combine column means, subtraction, scaling, and broadcasting in one final multi-step exercise.

**Visible inputs:** `x`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `normalized`

**Task:** Compute every named intermediate and normalize each feature column.

**Ingredients:** Use `mean = x.mean(dim=0, keepdim=True)`, `centered = x - mean`, `variance = (centered ** 2).mean(dim=0, keepdim=True)`, `inv_std = (variance + 1e-5).rsqrt()`, and `normalized = centered * inv_std`. This uses population variance and epsilon `1e-5`.

**Axis meaning in this exercise:** For `x`, axis `0` means examples and axis `1` means features; each column is one feature.

**New operation or idea explained:** **Normalization** recenters and rescales values. `mean(dim=0, keepdim=True)` averages examples while keeping a size-one example axis. **Variance** is the mean squared distance from the mean. **Epsilon** (`1e-5`) prevents division by zero. `rsqrt()` computes one divided by the square root.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Reduction:** an operation such as `mean` or `sum` that combines several values into fewer values.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Final practice: attention masking


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [3.0, 6.0, 9.0], [4.0, 8.0, 12.0]], dtype=DTYPE)
_register_case("ex043", {'x': x})


In [ ]:
# Exercise 043: complete only the fields requested above; do not use autograd.
# Define `ex043_x_shape`.
# Define `ex043_mean_shape`.
# Define `ex043_mean` with PyTorch tensor operations.
# Define `ex043_centered_shape`.
# Define `ex043_centered` with PyTorch tensor operations.
# Define `ex043_variance_shape`.
# Define `ex043_variance` with PyTorch tensor operations.
# Define `ex043_inv_std_shape`.
# Define `ex043_inv_std` with PyTorch tensor operations.
# Define `ex043_normalized_shape`.
# Define `ex043_normalized` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex043_x_shape", "ex043", "x_shape")
_check_private_value("ex043_mean_shape", "ex043", "mean_shape")
_check_private_tensor("ex043_mean", "ex043", "mean")
_check_private_value("ex043_centered_shape", "ex043", "centered_shape")
_check_private_tensor("ex043_centered", "ex043", "centered")
_check_private_value("ex043_variance_shape", "ex043", "variance_shape")
_check_private_tensor("ex043_variance", "ex043", "variance")
_check_private_value("ex043_inv_std_shape", "ex043", "inv_std_shape")
_check_private_tensor("ex043_inv_std", "ex043", "inv_std")
_check_private_value("ex043_normalized_shape", "ex043", "normalized_shape")
_check_private_tensor("ex043_normalized", "ex043", "normalized")


### Exercise 044 — Final practice: attention masking

**Purpose:** Prepare a causal mask and apply it across batch and head axes.

**Visible inputs:** `scores`, `causal`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `masked_scores`

**Task:** Create the mask view and masked scores.

**Ingredients:** Use `mask_view = causal[None, None, :, :]` and `masked_scores = scores.masked_fill(~mask_view, -1e9)`. Axes are `(batch, heads, query, key)`.

**Axis meaning in this exercise:** `scores` uses `(batch, heads, query, key)`; `causal` uses `(query, key)`.

**New operation or idea explained:** A **causal mask** prevents a query position from using future key positions. `masked_fill(~mask_view, -1e9)` puts a very negative value wherever attention is forbidden.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Final practice: pairwise squared distances


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
scores = torch.arange(64, dtype=DTYPE).reshape(2, 2, 4, 4)
causal = torch.tensor([[True, False, False, False], [True, True, False, False], [True, True, True, False], [True, True, True, True]], dtype=torch.bool)
_register_case("ex044", {'scores': scores, 'causal': causal})


In [ ]:
# Exercise 044: complete only the fields requested above; do not use autograd.
# Define `ex044_scores_shape`.
# Define `ex044_causal_shape`.
# Define `ex044_mask_view_shape`.
# Define `ex044_mask_view` with PyTorch tensor operations.
# Define `ex044_masked_scores_shape`.
# Define `ex044_masked_scores` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex044_scores_shape", "ex044", "scores_shape")
_check_private_value("ex044_causal_shape", "ex044", "causal_shape")
_check_private_value("ex044_mask_view_shape", "ex044", "mask_view_shape")
_check_private_tensor("ex044_mask_view", "ex044", "mask_view")
_check_private_value("ex044_masked_scores_shape", "ex044", "masked_scores_shape")
_check_private_tensor("ex044_masked_scores", "ex044", "masked_scores")


### Exercise 045 — Final practice: pairwise squared distances

**Purpose:** Compare every pair of items, keep feature values separate, then sum across features.

**Visible inputs:** `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `sq_distance`

**Task:** Create item-axis views, compute differences, square, and sum the feature axis.

**Ingredients:** Use `x_rows = x[:, None, :]`, `y_rows = y[None, :, :]`, `difference = x_rows - y_rows`, and `sq_distance = (difference ** 2).sum(dim=2)`.

**Axis meaning in this exercise:** `x` uses `(x_items, features)` and `y` uses `(y_items, features)`; the final sum combines feature values.

**New operation or idea explained:** A **squared distance** between two vectors is found by subtracting their features, squaring each difference, and summing those squares across the feature axis.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Reduction:** an operation such as `mean` or `sum` that combines several values into fewer values.

**Order:** Fill the requested shape reasoning before calculating tensor values.

**Next concept:** Fresh-kernel mastery run


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
x = torch.tensor([[0.0, 0.0, 0.0], [1.0, 2.0, 3.0]], dtype=DTYPE)
y = torch.tensor([[1.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 3.0]], dtype=DTYPE)
_register_case("ex045", {'x': x, 'y': y})


In [ ]:
# Exercise 045: complete only the fields requested above; do not use autograd.
# Define `ex045_x_shape`.
# Define `ex045_y_shape`.
# Define `ex045_x_rows_shape`.
# Define `ex045_x_rows` with PyTorch tensor operations.
# Define `ex045_y_rows_shape`.
# Define `ex045_y_rows` with PyTorch tensor operations.
# Define `ex045_difference_shape`.
# Define `ex045_difference` with PyTorch tensor operations.
# Define `ex045_sq_distance_shape`.
# Define `ex045_sq_distance` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex045_x_shape", "ex045", "x_shape")
_check_private_value("ex045_y_shape", "ex045", "y_shape")
_check_private_value("ex045_x_rows_shape", "ex045", "x_rows_shape")
_check_private_tensor("ex045_x_rows", "ex045", "x_rows")
_check_private_value("ex045_y_rows_shape", "ex045", "y_rows_shape")
_check_private_tensor("ex045_y_rows", "ex045", "y_rows")
_check_private_value("ex045_difference_shape", "ex045", "difference_shape")
_check_private_tensor("ex045_difference", "ex045", "difference")
_check_private_value("ex045_sq_distance_shape", "ex045", "sq_distance_shape")
_check_private_tensor("ex045_sq_distance", "ex045", "sq_distance")


## Completion standard

You have mastered this notebook when you can:

- distinguish scalar, vector, matrix, and higher-rank shapes;
- explain broadcasting first as value reuse;
- use singleton dimensions to express row, column, feature, batch, token, and channel intent;
- right-align shapes when debugging compatibility;
- predict result shapes without trial-and-error reshaping;
- diagnose invalid operations and repair them deliberately;
- use `expand`, `broadcast_to`, `repeat`, and `clone` for the intended storage behavior;
- apply broadcasting in common machine-learning tensor layouts.

Manual gradient derivation is intentionally omitted from this beginner workbook. Continue with the paired tensor-shapes/manual-backpropagation notebook after broadcasting itself is comfortable.

## Official references

- [PyTorch broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- [`Tensor.unsqueeze`](https://docs.pytorch.org/docs/stable/generated/torch.unsqueeze.html)
- [`Tensor.expand`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.expand.html)
- [`torch.broadcast_to`](https://docs.pytorch.org/docs/stable/generated/torch.broadcast_to.html)
- [`Tensor.repeat`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
